In [0]:
cust =  spark.read.format('delta').load('/Volumes/data/orders/files/chocolate/customers')
pro = spark.read.format('delta').load('/Volumes/data/orders/files/chocolate/products')
sto = spark.read.format('delta').load('/Volumes/data/orders/files/chocolate/stores')
sale = spark.read.format('delta').load('/Volumes/data/orders/files/chocolate/sales')
import pyspark.sql.functions as f
from pyspark.sql.window import Window

In [0]:
cust.printSchema()
pro.printSchema()
sto.printSchema()
sale.printSchema()

Top 5 most sold products

In [0]:
df = sale.join(pro, "product_id")\
    .groupBy("product_id", "product_name").agg(f.sum("quantity").alias("total")) \
    .orderBy(f.desc("total")).limit(5)
display(df)

Most popular product category per store

In [0]:
window = Window.partitionBy("store_id").orderBy(f.desc("total_quantity"))

df = sale.join(pro, "product_id").join(sto, "store_id") \
    .groupBy("store_id", "store_name", "category") \
    .agg(f.sum("quantity").alias("total_quantity")) \
    .withColumn("rank", f.row_number().over(window)) \
    .filter(f.col("rank") == 1) \
    .select("store_id", "store_name", "category", "total_quantity")

display(df)

Repeat customers (placed more than 1 order)

In [0]:
df = sale.groupBy("customer_id")\
    .agg(f.countDistinct("order_id").alias("order_count")) \
    .filter(f.col("order_count") > 1) \
    .join(cust, "customer_id") \
    .select("customer_id", "order_count")
display(df)

Find total sales (quantity) per store.

In [0]:
df = sale.join(sto, "store_id")\
    .groupBy("store_id", "store_name").agg(f.sum("quantity").alias("total_quantity")) \
    .select("store_id", "store_name", "total_quantity")
display(df)

Combine all tables to create a full order dataset.

In [0]:
df = sale.join(cust, "customer_id").join(pro, "product_id").join(sto, "store_id") \
    .select("order_id", "order_date", "customer_id", "product_id", "product_name", "category",
        "store_id", "store_name", "quantity", "unit_price",'discount','new_revenue','new_profit')
display(df)

Find top 5 most sold products by each store.

In [0]:
window = Window.partitionBy("store_id").orderBy(f.desc("total_quantity"))

df = sale.join(pro, "product_id").join(sto, "store_id") \
    .groupBy("store_id", "store_name", "product_id", "product_name") \
    .agg(f.sum("quantity").alias("total_quantity")) \
    .withColumn("rank", f.row_number().over(window)) \
    .filter(f.col("rank") <= 5) \
    .select("store_id", "store_name", "product_id", "product_name", "total_quantity")

display(df)

Count number of orders per city.

In [0]:
df = sale.join(sto, "store_id") \
    .groupBy("city") \
    .agg(f.countDistinct("order_id").alias("order_count")) \
    .select("city", "order_count")
display(df)

Assign row numbers to orders per customer based on date.

In [0]:
window = Window.partitionBy("customer_id").orderBy("order_date")

df = sale.withColumn("rn", f.row_number().over(window)) \
    .select("order_id", "customer_id", "order_date", "rn")
display(df)

Find previous order date for each customer (LAG).

In [0]:
window = Window.partitionBy("customer_id").orderBy("order_date")

df = sale.withColumn("prev_date", f.lag("order_date").over(window)) \
    .select("order_id", "customer_id", "order_date", "prev_date")
display(df)

Rank products based on total quantity sold.

In [0]:
window = Window.orderBy(f.desc("total_quantity"))

df = sale.join(pro, "product_id") \
    .groupBy("product_id", "product_name") \
    .agg(f.sum("quantity").alias("total_quantity")) \
    .withColumn("rank", f.rank().over(window)) \
    .select("product_id", "product_name", "total_quantity", "rank")

display(df)

Find top 3 products per category.

In [0]:
window = Window.partitionBy("category").orderBy(f.desc("total_quantity"))

df = sale.join(pro, "product_id") \
    .groupBy("category", "product_id", "product_name") \
    .agg(f.sum("quantity").alias("total_quantity")) \
    .withColumn("rank", f.row_number().over(window)) \
    .filter(f.col("rank") <= 3) \
    .select("category", "product_id", "product_name", "total_quantity")

display(df)

Calculate running total of sales per product.

In [0]:
window = Window.partitionBy("product_id").orderBy("order_date")\
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = sale.withColumn("running_total", f.sum("quantity").over(window)) \
    .select("order_id", "product_id", "order_date", "quantity", "running_total")
display(df)

Rank stores based on performance.

In [0]:
window = Window.orderBy(f.desc("total_sales"))

df = sale.groupBy("store_id") \
    .agg(f.sum("new_profit").alias("total_sales")) \
    .withColumn("rank", f.rank().over(window)) \
    .select("store_id", "total_sales", "rank")
display(df)